# 03 — The Measurement Loop

Before adding more building blocks (fuzzy, blocking, blended scores), we name the habit that ties everything together: **measure → change one thing → measure again → compare.** Don't guess whether a change helps—run it, evaluate against ground truth, and compare to the previous run. We'll use exact matching as the vehicle here; you'll use the same loop for fuzzy thresholds, blocking keys, and blend cutoffs.

**Back:** [02 Exact Matching](02_exact_matching.ipynb) · **Next:** [04 Fuzzy Matching](04_fuzzy_matching.ipynb)

## 1. The loop

1. Run matcher with your current setup (rules, and later: blocking, fuzzy, blend).
2. Evaluate against ground truth: `results.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")`.
3. Change **one** thing (e.g. add a rule, try a different blocking key, raise or lower a threshold).
4. Re-run and evaluate again.
5. Compare. Did precision or recall improve? Your data decides.

Repeat until you're satisfied. No magic—just measure, tune, compare.

## 2. Load data and ground truth

Same **evaluation set** as in 02: 50×50, 30 perfect pairs. We need a stable yardstick so every run is comparable.

In [ ]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = Path.cwd() if (Path.cwd() / "tutorial_data").exists() else Path.cwd().parent.parent / "docs" / "tutorial"
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_evaluation

left, right, ground_truth = load_evaluation()
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(f"Ground truth: {ground_truth.shape[0]} pairs. We'll compare every run against this.")

## 3. Run A: one rule

First run: email only. Run, evaluate, note the numbers.

In [ ]:
results_a = matcher.match(rules="email")
metrics_a = results_a.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Strategy A (email only): precision={metrics_a['precision']:.2%}, recall={metrics_a['recall']:.2%}, F1={metrics_a['f1']:.2%}")
print(f"Matches: {results_a.count}")

## 4. Run B: change one thing

Change one thing: add a cascading rule (email, then first_name+last_name for unmatched). Re-run, evaluate again.

In [ ]:
results_b = matcher.match(rules=["email", ["first_name", "last_name"]])
metrics_b = results_b.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print(f"Strategy B (email then name): precision={metrics_b['precision']:.2%}, recall={metrics_b['recall']:.2%}, F1={metrics_b['f1']:.2%}")
print(f"Matches: {results_b.count}")

## 5. Compare

Same ground truth, two strategies. You see the trade-off in numbers: did adding the name rule improve recall? Did it hurt precision? On your own data, this comparison tells you whether the change was worth it.

In [ ]:
print("A (email only):     ", f"precision={metrics_a['precision']:.2%}, recall={metrics_a['recall']:.2%}, F1={metrics_a['f1']:.2%}")
print("B (email + name):   ", f"precision={metrics_b['precision']:.2%}, recall={metrics_b['recall']:.2%}, F1={metrics_b['f1']:.2%}")

On this sample, email-only often gets full recall; cascading can add matches outside ground truth, so precision may dip. The point isn't the numbers—it's the **habit**: every time you change something (rules, blocking key, fuzzy threshold, blend cutoff), run the loop. Measure, tune, compare.

**Next:** [04 Fuzzy Matching](04_fuzzy_matching.ipynb) — you'll use this same loop to choose a fuzzy threshold and to compare exact vs fuzzy vs combo.